<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module34a/Lab07.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 7 — Running H$_2$ on Real Quantum Hardware

**Maps to:** Module 3, Lesson 6 + Module 4 (the "VQE is expensive" slide, made concrete)

**Time:** ~60 minutes, of which perhaps 15 are spent waiting in a queue
(instructor walkthrough ~15 min)

---

### The question this lab answers

Lab 5 got $-1.1373$ Ha on a perfect simulator. A real superconducting processor has
two-qubit error rates near $10^{-3}$, readout errors near $10^{-2}$, and qubits that
decohere in a couple hundred microseconds. **How far off is the answer, and what can you
do about it?**

### After this lab you can
1. Connect to IBM Quantum from a notebook and pick a backend.
2. Transpile a circuit to a specific device (ISA circuit) and re-map the observable to
   match — the step everyone forgets.
3. Rehearse on a noise model before spending queue time.
4. Run the H$_2$ energy estimate on hardware with `EstimatorV2`.
5. Compare raw vs. mitigated results (readout mitigation, dynamical decoupling, gate
   twirling, ZNE) and report the error in mHa and eV.

> **Before you start:** create a free IBM Quantum account at
> `https://quantum.cloud.ibm.com`, then copy your API token and instance (CRN) from the
> dashboard. Cells marked **[HARDWARE]** submit jobs; everything else runs offline.

In [ ]:
# %pip install -q qiskit qiskit-aer qiskit-ibm-runtime matplotlib scipy
import numpy as np, time
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

HARTREE_TO_EV = 27.2114
E_NUC = 0.71510

# The H2 problem from Lab 5, restated so this notebook stands alone.
H = SparsePauliOp.from_list([("II", -1.0523732458), ("IZ", +0.3979374248),
                             ("ZI", -0.3979374248), ("ZZ", -0.0112801043),
                             ("XX", +0.1809311998)])

def h2_ansatz(theta):
    qc = QuantumCircuit(2)
    qc.x(0)
    qc.s(1); qc.h(1); qc.h(0)
    qc.cx(0, 1); qc.rz(2*theta, 1); qc.cx(0, 1)
    qc.h(1); qc.sdg(1); qc.h(0)
    return qc

THETA_STAR = 0.1128
E_IDEAL = float(AerEstimator(options={"default_precision": 0.0}).run(
    [(h2_ansatz(THETA_STAR), H)]).result()[0].data.evs) + E_NUC
print(f"ideal (noiseless) energy at theta* : {E_IDEAL:.6f} Ha = {E_IDEAL*HARTREE_TO_EV:.3f} eV")

## 1. Connecting

Run `save_account` **once** (it writes `~/.qiskit/qiskit-ibm.json`); afterwards
`QiskitRuntimeService()` finds your credentials automatically.

On Colab the file disappears when the runtime resets, so either re-run `save_account`
each session or pass the token directly.

In [ ]:
#SKIP-VERIFY
from qiskit_ibm_runtime import QiskitRuntimeService

# --- Run ONCE, then comment out again -------------------------------------
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="PASTE_YOUR_API_TOKEN_HERE",
#     instance="PASTE_YOUR_INSTANCE_CRN_HERE",
#     overwrite=True, set_as_default=True)
# --------------------------------------------------------------------------

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)

print("backend      :", backend.name)
print("qubits       :", backend.num_qubits)
print("basis gates  :", backend.configuration().basis_gates)
print("pending jobs :", backend.status().pending_jobs)

### Reading the calibration data

Your Module 3 recap slide shows a device panel: 2Q error, readout error, $T_1$, $T_2$.
Pull the same numbers programmatically and use them to *predict* how bad your result will
be, before you run anything.

In [ ]:
#SKIP-VERIFY
props = backend.properties()
print(f"{'qubit':>6}{'T1 (us)':>10}{'T2 (us)':>10}{'readout err':>14}")
for q in range(min(5, backend.num_qubits)):
    print(f"{q:>6}{props.t1(q)*1e6:>10.1f}{props.t2(q)*1e6:>10.1f}{props.readout_error(q):>14.4f}")

cx_errors = [(pair, props.gate_error("cz", pair) if "cz" in
              backend.configuration().basis_gates else props.gate_error("cx", pair))
             for pair in backend.coupling_map.get_edges()[:5]]
for pair, err in cx_errors:
    print(f"2Q error on {pair}: {err:.2e}")

## 2. ISA circuits: two things must be translated, not one

A device only speaks its own basis gates on its own connectivity graph, so the circuit
must be transpiled. **The observable must follow.** After transpilation your logical
qubits 0 and 1 may live on physical qubits 43 and 44 of a 127-qubit chip — so the 2-qubit
operator `SparsePauliOp("XX")` has to be padded and permuted to a 127-qubit operator.
`observable.apply_layout(isa_circuit.layout)` does exactly that.

Forgetting this is the single most common hardware bug in VQE code, and it fails loudly
(dimension mismatch) rather than silently, which is a mercy.

In [ ]:
#SKIP-VERIFY
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_circuit = pm.run(h2_ansatz(THETA_STAR))
isa_H = H.apply_layout(isa_circuit.layout)

print("logical depth :", h2_ansatz(THETA_STAR).depth())
print("ISA depth     :", isa_circuit.depth())
print("ISA 2Q gates  :", sum(v for k, v in isa_circuit.count_ops().items() if k in ("cz", "cx", "ecr")))
print("physical qubits used:", isa_circuit.layout.initial_index_layout(filter_ancillas=True))
print("observable now acts on", isa_H.num_qubits, "qubits")

## 3. Rehearsal: a noise model, offline

Before you queue, run the identical pipeline against a *fake backend* — a snapshot of a
real device's calibration data, with the noise but without the wait. This runs on your
laptop and catches every bug except the ones caused by the queue.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

fake = FakeSherbrooke()
pm_fake = generate_preset_pass_manager(optimization_level=2, backend=fake)
isa_fake = pm_fake.run(h2_ansatz(THETA_STAR))
isa_H_fake = H.apply_layout(isa_fake.layout)

print("fake backend:", fake.name, "| ISA depth:", isa_fake.depth(),
      "| 2Q gates:", sum(v for k, v in isa_fake.count_ops().items() if k in ("cz","cx","ecr")))

noisy_est = AerEstimator.from_backend(fake, options={"default_precision": 0.003})
vals = [float(noisy_est.run([(isa_fake, isa_H_fake)]).result()[0].data.evs) + E_NUC
        for _ in range(5)]
vals = np.array(vals)

print(f"\nideal                 : {E_IDEAL:+.5f} Ha")
print(f"simulated noisy (x5)  : {vals.mean():+.5f} +/- {vals.std():.5f} Ha")
print(f"bias                  : {1000*(vals.mean()-E_IDEAL):+.2f} mHa "
      f"= {(vals.mean()-E_IDEAL)*HARTREE_TO_EV*1000:+.1f} meV")
print(f"\nchemical accuracy is 1.6 mHa -- compare.")

### Exercise 1 — predict before you measure

Using the noise model result and the device calibration, answer in the cell below:

* Is the noisy energy **above** or **below** the ideal one? Is that what the variational
  principle would lead you to expect, and why does noise break that guarantee?
* Roughly what fraction of the error would you attribute to readout, given that the
  dominant Hamiltonian terms are $Z$-type?

In [ ]:
# Decompose the error: which Pauli terms drift most under noise?
ideal_est = AerEstimator(options={"default_precision": 0.0})
print(f"{'term':>6}{'coeff':>12}{'ideal <P>':>12}{'noisy <P>':>12}{'contribution err (mHa)':>24}")
for pauli, coeff in zip(H.paulis, H.coeffs):
    term = SparsePauliOp(pauli)
    ideal_v = float(ideal_est.run([(h2_ansatz(THETA_STAR), term)]).result()[0].data.evs)
    noisy_v = float(noisy_est.run([(isa_fake, term.apply_layout(isa_fake.layout))]
                                  ).result()[0].data.evs)
    print(f"{pauli.to_label():>6}{coeff.real:>12.5f}{ideal_v:>12.4f}{noisy_v:>12.4f}"
          f"{1000*coeff.real*(noisy_v-ideal_v):>24.2f}")

## 4. On the real machine

`EstimatorV2` does the measurement-basis rotation, the shot sampling, and the weighted
sum for you. What you choose is **how much error suppression to pay for**:

| `resilience_level` | what it turns on | roughly how much slower |
|---|---|---|
| 0 | nothing | 1× |
| 1 | readout (TREX) mitigation | ~1–2× |
| 2 | + zero-noise extrapolation (ZNE) | ~3–5× |

Two cheap extras are worth enabling by hand:

* **Dynamical decoupling** — pulse sequences that keep idle qubits from dephasing. Nearly
  free.
* **Gate twirling** — randomizes coherent errors into stochastic ones, which mitigation
  handles far better.

In [ ]:
#SKIP-VERIFY
from qiskit_ibm_runtime import EstimatorV2, Batch

results = {}

with Batch(backend=backend) as batch:
    for level in [0, 1, 2]:
        est = EstimatorV2(mode=batch)
        est.options.resilience_level = level
        est.options.default_shots = 4000
        est.options.dynamical_decoupling.enable = True
        est.options.dynamical_decoupling.sequence_type = "XY4"
        est.options.twirling.enable_gates = True
        est.options.twirling.enable_measure = True
        job = est.run([(isa_circuit, isa_H)])
        results[level] = job
        print(f"submitted resilience_level={level}: job {job.job_id()}")

energies = {}
for level, job in results.items():
    r = job.result()[0]
    e = float(r.data.evs) + E_NUC
    std = float(r.data.stds) if hasattr(r.data, "stds") else float("nan")
    energies[level] = e
    print(f"level {level}: E = {e:+.5f} +/- {std:.5f} Ha   "
          f"error = {1000*(e-E_IDEAL):+7.2f} mHa   ({(e-E_IDEAL)*HARTREE_TO_EV*1000:+7.1f} meV)")

### Exercise 2 — report your numbers

Fill in the table below with **your** hardware results and comment. (If the queue is long,
use the noise-model numbers from Section 3 and say so.)

| run | E (Ha) | E (eV) | error vs ideal (mHa) | within chemical accuracy? |
|---|---|---|---|---|
| noiseless simulator | −1.13729 | −30.95 | 0.00 | — |
| noise model (FakeSherbrooke) | | | | |
| hardware, resilience 0 | | | | |
| hardware, resilience 1 | | | | |
| hardware, resilience 2 | | | | |

In [ ]:
#SKIP-VERIFY
labels = ["ideal", "noisy sim", "hw L0", "hw L1", "hw L2"]
values = [E_IDEAL, vals.mean(), energies[0], energies[1], energies[2]]

plt.figure(figsize=(6.4, 3.6))
plt.bar(labels, [1000*(v - E_IDEAL) for v in values], color=["k","gray","C3","C1","C2"])
plt.axhline(1.6, color="r", ls=":", label="chemical accuracy (1.6 mHa)")
plt.axhline(-1.6, color="r", ls=":")
plt.ylabel("error vs ideal (mHa)"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## 5. A whole scan in one job

Estimator V2 takes a **list of PUBs** (circuit, observable, parameter values). Submitting
five points as one job is far faster than five jobs, because you queue once. This is the
practical shape of a hardware VQE: batch aggressively.

In [ ]:
#SKIP-VERIFY
from qiskit.circuit import Parameter

theta = Parameter("theta")
param_circ = QuantumCircuit(2)
param_circ.x(0); param_circ.s(1); param_circ.h(1); param_circ.h(0)
param_circ.cx(0, 1); param_circ.rz(2*theta, 1); param_circ.cx(0, 1)
param_circ.h(1); param_circ.sdg(1); param_circ.h(0)

isa_param = pm.run(param_circ)
isa_H_param = H.apply_layout(isa_param.layout)

scan = np.linspace(-0.4, 0.6, 9)
est = EstimatorV2(mode=backend)
est.options.resilience_level = 1
est.options.default_shots = 4000
est.options.dynamical_decoupling.enable = True

job = est.run([(isa_param, isa_H_param, scan.reshape(-1, 1))])
print("job:", job.job_id())
res = job.result()[0]
E_hw = np.array(res.data.evs) + E_NUC

E_ref = [float(ideal_est.run([(h2_ansatz(t), H)]).result()[0].data.evs) + E_NUC for t in scan]
plt.figure(figsize=(6.4, 3.6))
plt.plot(scan, E_ref, "k-", label="ideal")
plt.plot(scan, E_hw, "o-", label=f"{backend.name}, mitigated")
plt.xlabel(r"$\theta$"); plt.ylabel("E (Ha)"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"hardware minimum: theta = {scan[np.argmin(E_hw)]:.3f}, E = {E_hw.min():.5f} Ha")

### Exercise 3 — the landscape is the point

Look at your scan. The ideal well is about **20 mHa** deep. Compare that to your
hardware error bars.

* Could a classical optimizer, seeing only your hardware curve, find $\theta^*$?
* Does noise shift the *location* of the minimum, or only its *depth*? Which of those
  two failures is more forgivable, and why? (Hint: think about what you report at the
  end — a geometry or an energy?)

## 6. What this costs, honestly

You just ran **one** energy evaluation at **one** geometry, for a **two-qubit**
molecule. Put that next to the arithmetic on the last Module 3 slide.

In [ ]:
n_settings, shots, evals, geometries = 3, 4000, 30, 20
per_energy = n_settings * shots
print(f"per energy evaluation : {per_energy:,} shots")
print(f"per inner loop        : {per_energy*evals:,} shots")
print(f"full dissociation curve: {per_energy*evals*geometries:,} shots")
print(f"\nAt ~5,000 shots/second including overhead: "
      f"{per_energy*evals*geometries/5000/60:.1f} minutes of QPU time -- for H2.")
print("Scale the same estimate to LiH (136 settings, ~90 parameters, ~900 evaluations):")
print(f"  {136*shots*900*geometries:,} shots = "
      f"{136*shots*900*geometries/5000/3600:,.0f} hours.")
print("\nThis is why measurement grouping, shot allocation, shallow ansatze, and")
print("error mitigation are the active research frontier -- not the algorithm itself.")

## Checkpoint

1. Why must you call `apply_layout` on the observable, and what error do you get if you
   do not?
2. Name one error source dynamical decoupling addresses and one it does not.
3. Your hardware energy came out *below* the true ground state. Give two possible causes,
   one statistical and one systematic.
4. `resilience_level=2` costs several times more shots. Under what circumstances is that
   a good trade, and when is it wasted?
5. You have 10 minutes of QPU time. Would you spend it on more shots at one geometry, or
   fewer shots at more geometries? Justify in terms of what you are trying to report.

### What is next
Labs 8 and 9 take VQE out of chemistry: the same machinery applied to **materials
models** — magnetism in Lab 8, correlated electrons in metals in Lab 9.